# Prepare Data

This notebook preprocesses data, extracts features and stores the preprocessed data. The saved data is 2.63 GB.

Also provides a minimalist training pipeline of the model for testing.

In [1]:
import os
import sys

sys.path.append("..")

import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
import uproot
import awkward as ak
from sklearn.model_selection import train_test_split


# Import our modules
from src.data_utils import (
    save_datasets_hdf5,
    load_datasets_hdf5
)

from src.features.global_features import extract_global_features
from src.features.particle import extract_and_pad_particle_features
from src.features.vertex import extract_and_pad_vertex_features
from src.model_utils.model import build_classifier

# Set matplotlib style
plt.style.use("hep.mplstyle")

# Check TensorFlow version and GPU availability
print(f"TensorFlow version: {tf.__version__}")
print(f"GPU available: {tf.config.list_physical_devices('GPU')}")

# Set random seeds for reproducibility
SEED = 42
tf.random.set_seed(SEED)
np.random.seed(SEED)

# File paths
DATA_DIR = "../datasets"

FILE_LIST = [
    f"{DATA_DIR}/Zbb.root",
    f"{DATA_DIR}/Zcc.root",
    f"{DATA_DIR}/Zss.root"
]
FILE_LABEL_MAP = {
    FILE_LIST[0]: 0,  # b
    FILE_LIST[1]: 1,  # c
    FILE_LIST[2]: 2   # s
}


TensorFlow version: 2.18.0
GPU available: []


## File Check

In [2]:
# Check if all files exist in DATA_DIR
missing_files = []
for file in FILE_LIST:
    file_path = os.path.join(DATA_DIR, file)
    if not os.path.isfile(file_path):
        missing_files.append(file)

if missing_files:
    raise FileNotFoundError(f"The following files are missing in {DATA_DIR}: {', '.join(missing_files)}")
else:
    print(f"All required files exist in {DATA_DIR}")

All required files exist in ../datasets


In [3]:
# events = load_root_file(f"{DATA_DIR}/Zbb.root")

all_events = []
all_labels = []
for filepath, label in FILE_LABEL_MAP.items():
    events = uproot.open(f"{filepath}:events").arrays(library="ak")
    labels = ak.Array(np.full(len(events), label, dtype=np.int64))
    events = ak.with_field(events, labels, "label")
    all_events.append(events)

## 2. Features



### Dataset Split

In [4]:
combined_events = ak.concatenate(all_events)
# combined_events = combined_events[:50] # For testing
labels = combined_events["label"]

event_indices = np.arange(len(combined_events))

train_indices, temp_indices = train_test_split(
    event_indices, 
    test_size=0.3,
    stratify=labels,
    random_state=42
)
val_indices, test_indices = train_test_split(
    temp_indices, 
    test_size=0.5,
    stratify=labels[temp_indices],
    random_state=42
)

train_events = combined_events[train_indices]
val_events = combined_events[val_indices]
test_events = combined_events[test_indices]

print(f"Training set size: {len(train_events)} ({len(train_events)/len(combined_events)*100:.1f}%)")
print(f"Validation set size: {len(val_events)} ({len(val_events)/len(combined_events)*100:.1f}%)")
print(f"Test set size: {len(test_events)} ({len(test_events)/len(combined_events)*100:.1f}%)")
print(train_events.fields)

Training set size: 427147 (70.0%)
Validation set size: 91531 (15.0%)
Test set size: 91532 (15.0%)
['Thrust_x', 'Thrust_y', 'Thrust_z', 'Thrust_xerr', 'Thrust_yerr', 'Thrust_zerr', 'nParticle', 'Particle_e', 'Particle_m', 'Particle_q', 'Particle_p', 'Particle_pt', 'Particle_eta', 'Particle_phi', 'Particle_px', 'Particle_py', 'Particle_pz', 'Particle_ID', 'Particle_orivtx_ind', 'nVertex', 'Vertex_ntracks', 'Vertex_chi2', 'Vertex_isPV', 'Vertex_m', 'Vertex_x', 'Vertex_y', 'Vertex_z', 'Vertex_xerr', 'Vertex_yerr', 'Vertex_zerr', 'label']


### Prepare Data

Standardisation is implemented in the feature extraction functions. 

The required statistics are calculated from the training data, and these will be used to standardise the validation and test data.

In [ ]:
train_global_features, global_stats = extract_global_features(train_events, apply_scaling=True, scaling_stats=None)
print(train_global_features.keys())
print(train_global_features["thrust_x"].shape)
print(global_stats)

dict_keys(['thrust_x', 'thrust_y', 'thrust_z', 'n_particles', 'n_vertices', 'n_secondary_vertices', 'n_charged', 'total_energy', 'total_pt', 'pv_ntracks', 'pv_chi2'])
(427147,)
{'thrust_x': (-0.0005352632943975092, 0.5501359770285853), 'thrust_y': (-0.0003851744969325096, 0.5492388891448968), 'thrust_z': (0.000629508091087853, 0.6290359434076567), 'n_particles': (43.07257454693583, 12.5410516841747), 'n_vertices': (3.4025335540223858, 1.6567437184235334), 'n_secondary_vertices': (2.4025335540223858, 1.6567437184235334), 'n_charged': (19.88263993426151, 6.114830560851765), 'total_energy': (87.17340711797063, 9.388059305688154), 'total_pt': (64.86706603061695, 18.507416613264823), 'pv_ntracks': (13.012548373276648, 5.781567464490091), 'pv_chi2': (115.93442516850598, 30499.648183476602)}


In [ ]:
train_particle_features, particle_stats = extract_and_pad_particle_features(train_events, apply_scaling=True, scaling_stats=None)
print(train_particle_features.keys())
print(train_particle_features["Particle_px"].shape)
print(particle_stats)

dict_keys(['Particle_px', 'Particle_py', 'Particle_pz', 'Particle_e', 'Particle_q', 'Particle_angle_to_thrust', 'is_charged', 'is_kaon', 'is_pion', 'is_muon', 'is_electron', 'is_photon', 'Vertex_isPV', 'Vertex_chi2', 'Vertex_ntracks', 'Vertex_m', 'Vertex_x', 'Vertex_y', 'Vertex_z', 'displacement_from_PV'])
(427147, 25)
{'Particle_px': (0.0017537796457489242, 2.4026476115008912), 'Particle_py': (0.0004680198885262255, 2.4165767497088106), 'Particle_pz': (2.1276672308352542e-05, 2.9707791936671546), 'Particle_pt': (2.008435667491871, 2.7529526319598023), 'Particle_e': (2.6764423731674687, 3.6555094035005538), 'Particle_q': (3.368117400685115e-05, 0.99999999943279), 'Particle_angle_to_thrust': (1.659085822759685, 1.1671620755370729), 'is_charged': (1.0, 1.0), 'is_kaon': (0.13106818357428582, 0.3374749099249055), 'is_pion': (0.8064463042836509, 0.39508310846060224), 'is_muon': (0.005467247039523867, 0.0737384312915096), 'is_electron': (0.0112552081991865, 0.10549184085785672), 'is_photon':

In [ ]:
train_vertex_features, vertex_stats = extract_and_pad_vertex_features(train_events, apply_scaling=True, scaling_stats=None)
print(train_vertex_features.keys())
print(train_vertex_features["sv_m"].shape)
print(vertex_stats)

dict_keys(['sv_m', 'sv_chi2', 'sv_ntracks', 'sv_x', 'sv_y', 'sv_z'])
(427147, 4)
{'Vertex_m': (0.9835591358079221, 0.6945320468748125), 'Vertex_chi2': (158.60831342843693, 14822.20903641836), 'Vertex_ntracks': (2.453099923506799, 0.9369349534603697), 'Vertex_x': (13.747546425310134, 13233.52251497124), 'Vertex_y': (-7.393930701007517, 5510.76831565774), 'Vertex_z': (-13633.703231449163, 13824947.906551328)}


In [8]:
train_labels = tf.convert_to_tensor(train_events.label, dtype=tf.float32)
print(train_labels.shape)
dataset_dict = {
    "global": np.stack([train_global_features[f] for f in train_global_features], axis=-1),
    "particle": np.stack([train_particle_features[f] for f in train_particle_features], axis=-1),
    "sv": np.stack([train_vertex_features[f] for f in train_vertex_features], axis=-1)
}

dataset_dict.keys()

(427147,)


dict_keys(['global', 'particle', 'sv'])

In [9]:
print(dataset_dict["global"].shape)            # (N_events, 11)
print(dataset_dict["particle"].shape)       # (N_events, 25, 20)
print(dataset_dict["sv"].shape)             # (N_events, 4, 6)

(427147, 11)
(427147, 25, 20)
(427147, 4, 6)


In [10]:
global_tensor = tf.convert_to_tensor(dataset_dict["global"], dtype=tf.float32)
particle_tensor = tf.convert_to_tensor(dataset_dict["particle"], dtype=tf.float32)
sv_tensor = tf.convert_to_tensor(dataset_dict["sv"], dtype=tf.float32)

particle_mask = tf.reduce_any(tf.not_equal(particle_tensor, -999.0), axis=-1)   # (N_events, 50)
sv_mask = tf.reduce_any(tf.not_equal(sv_tensor, -999.0), axis=-1)               # (N_events, 4)

### Apply to Validation & Test Sets

In [11]:
# Extract features
val_global_features, _ = extract_global_features(val_events, apply_scaling=True, scaling_stats=global_stats)
val_particle_features, _ = extract_and_pad_particle_features(val_events, apply_scaling=True, scaling_stats=particle_stats)
val_vertex_features, _ = extract_and_pad_vertex_features(val_events, apply_scaling=True, scaling_stats=vertex_stats)

# Prepare dataset dictionary
val_dataset_dict = {
    "global": np.stack([val_global_features[f] for f in val_global_features], axis=-1),  # (N_val_events, num_jet_features)
    "particle": np.stack([val_particle_features[f] for f in val_particle_features], axis=-1),  # (N_val_events, max_particles, num_charged_features)
    "sv": np.stack([val_vertex_features[f] for f in val_vertex_features], axis=-1)  # (N_val_events, max_vertices, num_sv_features)
}
val_labels = tf.convert_to_tensor(val_events.label, dtype=tf.float32)  # (N_val_events, )

# Convert to tensors
val_global_tensor = tf.convert_to_tensor(val_dataset_dict["global"], dtype=tf.float32)
val_particle_tensor = tf.convert_to_tensor(val_dataset_dict["particle"], dtype=tf.float32)
val_sv_tensor = tf.convert_to_tensor(val_dataset_dict["sv"], dtype=tf.float32)

val_particle_mask = tf.reduce_any(tf.not_equal(val_particle_tensor, -999.0), axis=-1)   # (N_events, 25)
val_sv_mask = tf.reduce_any(tf.not_equal(val_sv_tensor, -999.0), axis=-1)               # (N_events, 4)
print(val_global_tensor.shape)
print(val_particle_tensor.shape)
print(val_sv_tensor.shape)
print(val_labels.shape)
print("--------------------------------")
print(val_particle_mask.shape)
print(val_sv_mask.shape)

(91531, 11)
(91531, 25, 20)
(91531, 4, 6)
(91531,)
--------------------------------
(91531, 25)
(91531, 4)


In [12]:
# Extract features
test_global_features, _ = extract_global_features(test_events, apply_scaling=True, scaling_stats=global_stats)
test_particle_features, _ = extract_and_pad_particle_features(test_events, apply_scaling=True, scaling_stats=particle_stats)
test_vertex_features, _ = extract_and_pad_vertex_features(test_events, apply_scaling=True, scaling_stats=vertex_stats)

# Prepare dataset dictionary
test_dataset_dict = {
    "global": np.stack([test_global_features[f] for f in test_global_features], axis=-1),  # (N_test_events, num_jet_features)
    "particle": np.stack([test_particle_features[f] for f in test_particle_features], axis=-1),  # (N_test_events, max_particles, num_charged_features)
    "sv": np.stack([test_vertex_features[f] for f in test_vertex_features], axis=-1)  # (N_test_events, max_vertices, num_sv_features)
}
test_labels = tf.convert_to_tensor(test_events.label, dtype=tf.float32)  # (N_test_events, )

# Convert to tensors
test_global_tensor = tf.convert_to_tensor(test_dataset_dict["global"], dtype=tf.float32)
test_particle_tensor = tf.convert_to_tensor(test_dataset_dict["particle"], dtype=tf.float32)
test_sv_tensor = tf.convert_to_tensor(test_dataset_dict["sv"], dtype=tf.float32)

test_particle_mask = tf.reduce_any(tf.not_equal(test_particle_tensor, -999.0), axis=-1)   # (N_events, 25)
test_sv_mask = tf.reduce_any(tf.not_equal(test_sv_tensor, -999.0), axis=-1)               # (N_events, 4)

print(test_global_tensor.shape)
print(test_particle_tensor.shape)
print(test_sv_tensor.shape)
print(test_labels.shape)
print("--------------------------------")
print(test_particle_mask.shape)
print(test_sv_mask.shape)

(91532, 11)
(91532, 25, 20)
(91532, 4, 6)
(91532,)
--------------------------------
(91532, 25)
(91532, 4)


### Save Data

In [ ]:
datasets = {
    'train': {
        'global': np.stack([train_global_features[f] for f in train_global_features], axis=-1),
        'particle': np.stack([train_particle_features[f] for f in train_particle_features], axis=-1),
        'sv': np.stack([train_vertex_features[f] for f in train_vertex_features], axis=-1)
    },
    'val': {
        'global': np.stack([val_global_features[f] for f in val_global_features], axis=-1),
        'particle': np.stack([val_particle_features[f] for f in val_particle_features], axis=-1),
        'sv': np.stack([val_vertex_features[f] for f in val_vertex_features], axis=-1)
    },
    'test': {
        'global': np.stack([test_global_features[f] for f in test_global_features], axis=-1),
        'particle': np.stack([test_particle_features[f] for f in test_particle_features], axis=-1),
        'sv': np.stack([test_vertex_features[f] for f in test_vertex_features], axis=-1)
    }
}

labels = {
    'train': train_labels,
    'val': val_labels,
    'test': test_labels
}

masks = {
    'train': {'particle_mask': particle_mask, 'sv_mask': sv_mask},
    'val': {'particle_mask': val_particle_mask, 'sv_mask': val_sv_mask},
    'test': {'particle_mask': test_particle_mask, 'sv_mask': test_sv_mask}
}

# Save to disk
data_file = save_datasets_hdf5(datasets, labels, masks, output_dir="../datasets") # 2.63GB

In [15]:
# Get numpy arrays for inspection
np_datasets = load_datasets_hdf5(data_file, return_numpy=True)

for split in ['train', 'val', 'test']:
    print(f"Split: {split}")
    print(f"Global shape: {np_datasets[split]['global'].shape}")
    print(f"Particle shape: {np_datasets[split]['particle'].shape}")
    print(f"SV shape: {np_datasets[split]['sv'].shape}")
    print(f"Labels shape: {np_datasets[split]['labels'].shape}")

# Get tf.data.Dataset objects for training
tf_datasets = load_datasets_hdf5(data_file, return_numpy=False)
train_ds = tf_datasets['train']
val_ds = tf_datasets['val']
test_ds = tf_datasets['test']

Split: train
Global shape: (427147, 11)
Particle shape: (427147, 25, 20)
SV shape: (427147, 4, 6)
Labels shape: (427147,)
Split: val
Global shape: (91531, 11)
Particle shape: (91531, 25, 20)
SV shape: (91531, 4, 6)
Labels shape: (91531,)
Split: test
Global shape: (91532, 11)
Particle shape: (91532, 25, 20)
SV shape: (91532, 4, 6)
Labels shape: (91532,)


## 3. Test Training

In [ ]:
model = build_classifier()

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
    loss=tf.keras.losses.SparseCategoricalCrossentropy(from_logits=False),
    metrics=[tf.keras.metrics.SparseCategoricalAccuracy()]
)

model.fit(train_ds, epochs=1, validation_data=val_ds) # 1 epoch + validation takes ~ 8 mins (on my MacBook)

6675/6675 ━━━━━━━━━━━━━━━━━━━━ 509s 75ms/step - loss: 0.5609 - sparse_categorical_accuracy: 0.8463 - val_loss: 0.2812 - val_sparse_categorical_accuracy: 0.9036
